# Category Image Usage

Given a Wikimedia Commons category, this notebook:
1. Collects all images in the category
2. Looks up which English Wikipedia pages use each image
3. Shows a table of files + their usage, and a top-N ranking of pages
4. Saves both tables to CSV

Uses only the Wikimedia API — no scraping.

In [ ]:
# --- Config ---
CATEGORY = "Category:Images from Wiki Loves Africa 2025"  # change this
TOP_N = 5                                      # top pages to highlight
OUTPUT_FILE_USAGE = "file_usage.csv"
OUTPUT_PAGE_RANKING = "page_ranking.csv"

In [ ]:
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from IPython.display import display

API_COMMONS = "https://commons.wikimedia.org/w/api.php"
USER_AGENT = "WikimediaAnalysis/1.0 (personal research project; https://github.com/lgelauff/wikimedia-analysis)"

BATCH_SIZE = 50  # files per API request
WORKERS = 3      # concurrent batch requests

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

In [ ]:
def get_category_images(category: str) -> list[str]:
    """Return all File: titles in a Commons category."""
    if not category.startswith("Category:"):
        category = "Category:" + category

    files = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": category,
        "cmtype": "file",
        "cmlimit": "500",
        "format": "json",
    }
    while True:
        resp = session.get(API_COMMONS, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        members = data.get("query", {}).get("categorymembers", [])
        files.extend(m["title"] for m in members)
        if "continue" not in data:
            break
        params.update(data["continue"])
    return files


def _get_with_backoff(params: dict) -> dict:
    """GET from the Commons API, retrying on 429 with exponential backoff."""
    delay = 5
    while True:
        resp = session.get(API_COMMONS, params=params, timeout=30)
        if resp.status_code == 429:
            print(f"\n  Rate limited — waiting {delay}s...", end="\r")
            time.sleep(delay)
            delay = min(delay * 2, 60)
            continue
        resp.raise_for_status()
        return resp.json()


def get_enwiki_usage_batch(file_titles: list[str]) -> dict[str, list[str]]:
    """
    Fetch enwiki usage for up to 50 files in a single API request.
    Returns {file_title: [enwiki_page, ...]}.
    """
    result = {t: [] for t in file_titles}
    params = {
        "action": "query",
        "titles": "|".join(file_titles),
        "prop": "globalusage",
        "gulimit": "500",
        "format": "json",
    }
    while True:
        data = _get_with_backoff(params)
        for page in data.get("query", {}).get("pages", {}).values():
            title = page.get("title")
            if title in result:
                for u in page.get("globalusage", []):
                    if u.get("wiki") == "en.wikipedia.org":
                        result[title].append(u["title"])
        if "continue" not in data:
            break
        params.update(data["continue"])
    return result

In [ ]:
import json, os, re

# Normalise category prefix
if not CATEGORY.startswith("Category:"):
    CATEGORY = "Category:" + CATEGORY

# Cache path: tmp/category_files_<sanitized_name>.json
os.makedirs("tmp", exist_ok=True)
cache_key = re.sub(r"[^\w]+", "_", CATEGORY).strip("_")
cache_path = f"tmp/category_files_{cache_key}.json"

if os.path.exists(cache_path):
    with open(cache_path) as f:
        files = json.load(f)
    print(f"Loaded {len(files)} file(s) from cache ({cache_path})")
else:
    print(f"Fetching files from: {CATEGORY}")
    files = get_category_images(CATEGORY)
    with open(cache_path, "w") as f:
        json.dump(files, f)
    print(f"Found {len(files)} file(s). Cached to {cache_path}")

In [ ]:
# Fetch English Wikipedia usage — batched (50 files/request) + 5 concurrent workers
file_usage: dict[str, list[str]] = {}
page_images: dict[str, list[str]] = defaultdict(list)

batches = [files[i:i + BATCH_SIZE] for i in range(0, len(files), BATCH_SIZE)]
completed = 0
print(f"Fetching usage: {len(files)} files → {len(batches)} batches, {WORKERS} concurrent workers...")

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    future_to_batch = {executor.submit(get_enwiki_usage_batch, batch): batch for batch in batches}
    for future in as_completed(future_to_batch):
        batch_result = future.result()
        for file_title, pages in batch_result.items():
            file_usage[file_title] = pages
            for page in pages:
                page_images[page].append(file_title)
        completed += len(future_to_batch[future])
        print(f"  {completed}/{len(files)} files done", end="\r")

print(f"\nDone. {sum(1 for p in file_usage.values() if p)}/{len(files)} files used on English Wikipedia.")

In [ ]:
# --- File usage table ---
file_rows = [
    {
        "file": title,
        "pages_count": len(pages),
        "pages": " | ".join(pages),
    }
    for title, pages in file_usage.items()
]
df_files = pd.DataFrame(file_rows).sort_values("pages_count", ascending=False).reset_index(drop=True)

print("=== File usage ===")
display(df_files)

df_files.to_csv(OUTPUT_FILE_USAGE, index=False)
print(f"Saved to {OUTPUT_FILE_USAGE}")

In [ ]:
# --- Top-N pages table ---
page_rows = [
    {
        "rank": rank + 1,
        "page": page,
        "images_count": len(imgs),
        "images": " | ".join(imgs),
    }
    for rank, (page, imgs) in enumerate(
        sorted(page_images.items(), key=lambda kv: len(kv[1]), reverse=True)[:TOP_N]
    )
]
df_pages = pd.DataFrame(page_rows)

print(f"=== Top {TOP_N} English Wikipedia pages ===")
display(df_pages)

df_pages.to_csv(OUTPUT_PAGE_RANKING, index=False)
print(f"Saved to {OUTPUT_PAGE_RANKING}")